<a href="https://colab.research.google.com/github/HSA2005-cloud/BIS-Lab/blob/main/2.%20Particle%20Swarm%20Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Demand and supply (kW)
DR, DI = 600, 900
S = 1200
ER, EI = 400, 500

# Cost per kWh (₹)
cR, cI = 8, 10

# Penalty weights (illustrative)
wE, wN = 1000, 100

# PSO parameters
N = 30
T = 100
w, c1, c2 = 0.7, 1.5, 1.5

# Keep allocations within valid limits
def repair(x):
    x = np.clip(x, [0, 0], [DR, DI])

    if x.sum() > S:
        x = x * S / x.sum()

    return x

# Objective / fitness function
def fitness(x):
    PR, PI = x

    # Essential demand is served first
    URE = max(0, ER - PR)
    UIE = max(0, EI - PI)

    URN = max(0, DR - ER - max(0, PR - ER))
    UIN = max(0, DI - EI - max(0, PI - EI))

    supply_cost = cR * PR + cI * PI

    return (
        supply_cost
        + wE * (URE + UIE)
        + wN * (URN + UIN)
    )

# Initialize particles
positions = np.random.uniform(
    [0, 0], [DR, DI], (N, 2)
)
positions = np.array([repair(x) for x in positions])

velocities = np.zeros((N, 2))

pbest = positions.copy()
pbest_fit = np.array([fitness(x) for x in positions])

best_index = np.argmin(pbest_fit)
gbest = pbest[best_index].copy()
gbest_fit = pbest_fit[best_index]

# Main PSO loop
for _ in range(T):
    for i in range(N):

        r1 = np.random.random(2)
        r2 = np.random.random(2)

        # Velocity update
        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest[i] - positions[i])
            + c2 * r2 * (gbest - positions[i])
        )

        # Position update and constraint handling
        positions[i] = repair(
            positions[i] + velocities[i]
        )

        # Evaluate new allocation
        score = fitness(positions[i])

        # Update personal best
        if score < pbest_fit[i]:
            pbest[i] = positions[i].copy()
            pbest_fit[i] = score

    # Update global best
    best_index = np.argmin(pbest_fit)

    if pbest_fit[best_index] < gbest_fit:
        gbest = pbest[best_index].copy()
        gbest_fit = pbest_fit[best_index]

# Display the best solution found
PR, PI = gbest

print("Residential allocation:", round(PR, 2), "kW")
print("Industrial allocation:", round(PI, 2), "kW")
print("Total allocated:", round(PR + PI, 2), "kW")
print("Best fitness:", round(gbest_fit, 2))
print("Residential unmet demand:", round(DR - PR, 2), "kW")
print("Industrial unmet demand:", round(DI - PI, 2), "kW")

Residential allocation: 600.0 kW
Industrial allocation: 600.0 kW
Total allocated: 1200.0 kW
Best fitness: 40800.0
Residential unmet demand: 0.0 kW
Industrial unmet demand: 300.0 kW
